In [24]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder,OrdinalEncoder
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import pickle
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from scipy.stats import zscore
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline



## **Diabetes**

In [13]:
data = pd.read_csv('Data/diabetes.csv')

In [14]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Age                 520 non-null    int64 
 1   Gender              520 non-null    object
 2   Polyuria            520 non-null    object
 3   Polydipsia          520 non-null    object
 4   sudden weight loss  520 non-null    object
 5   weakness            520 non-null    object
 6   Polyphagia          520 non-null    object
 7   Genital thrush      520 non-null    object
 8   visual blurring     520 non-null    object
 9   Itching             520 non-null    object
 10  Irritability        520 non-null    object
 11  delayed healing     520 non-null    object
 12  partial paresis     520 non-null    object
 13  muscle stiffness    520 non-null    object
 14  Alopecia            520 non-null    object
 15  Obesity             520 non-null    object
 16  class               520 no

In [15]:
data

,Age,Gender,Polyuria,Polydipsia,sudden weight loss,weakness,Polyphagia,Genital thrush,visual blurring,Itching,Irritability,delayed healing,partial paresis,muscle stiffness,Alopecia,Obesity,class
0,40,Male,No,Yes,No,Yes,No,No,No,Yes,No,Yes,No,Yes,Yes,Yes,Positive
1,58,Male,No,No,No,Yes,No,No,Yes,No,No,No,Yes,No,Yes,No,Positive
2,41,Male,Yes,No,No,Yes,Yes,No,No,Yes,No,Yes,No,Yes,Yes,No,Positive
3,45,Male,No,No,Yes,Yes,Yes,Yes,No,Yes,No,Yes,No,No,No,No,Positive
4,60,Male,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,39,Female,Yes,Yes,Yes,No,Yes,No,No,Yes,No,Yes,Yes,No,No,No,Positive
516,48,Female,Yes,Yes,Yes,Yes,Yes,No,No,Yes,Yes,Yes,Yes,No,No,No,Positive
517,58,Female,Yes,Yes,Yes,Yes,Yes,No,Yes,No,No,No,Yes,Yes,No,Yes,Positive
518,32,Female,No,No,No,Yes,No,No,Yes,Yes,No,Yes,No,No,Yes,No,Negative


In [16]:
class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass  

    def cap_outliers(self, data, column_name):
        skewness = data[column_name].skew()

        if abs(skewness) < 1:  
            mean = data[column_name].mean()
            std_dev = data[column_name].std()
            lower_bound = mean - 3 * std_dev
            upper_bound = mean + 3 * std_dev
        else:  
            Q1 = data[column_name].quantile(0.25)
            Q3 = data[column_name].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

        data[column_name] = np.where(data[column_name] < lower_bound, lower_bound,
                                     np.where(data[column_name] > upper_bound, upper_bound, data[column_name]))
        return data

    def fit(self, X, y=None):
        return self  

    def transform(self, X):
        X = X.copy()
        numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        for column in numerical_cols:
            X = self.cap_outliers(X, column)
        return X

In [17]:
X = data.drop(columns=['class'])
y = data['class']

In [18]:
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns


In [19]:
numerical_features

Index(['Age'], dtype='object')

In [20]:
categorical_features

Index(['Gender', 'Polyuria', 'Polydipsia', 'sudden weight loss', 'weakness',
       'Polyphagia', 'Genital thrush', 'visual blurring', 'Itching',
       'Irritability', 'delayed healing', 'partial paresis',
       'muscle stiffness', 'Alopecia', 'Obesity'],
      dtype='object')

In [26]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
le = LabelEncoder()
le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)

In [27]:
models = {
    'Logistic Regression': (LogisticRegression(class_weight='balanced', max_iter=1000), {
        'classifier__C': [0.1, 1, 10]
    }),
    'Random Forest': (RandomForestClassifier(random_state=42), {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [None, 10, 20]
    }),
    'Decision Tree': (DecisionTreeClassifier(random_state=42), {
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 10, 20]
    }),
    'AdaBoost': (AdaBoostClassifier(random_state=42), {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__learning_rate': [0.01, 0.1, 1]
    })
}

best_models = {}

for model_name, (model, param_grid) in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    best_models[model_name] = grid_search.best_estimator_
    print(f"{model_name}: Best parameters: {grid_search.best_params_}")

for model_name, model in best_models.items():
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{model_name} Test Accuracy: {accuracy:.4f}")
    print(classification_report(y_test, y_pred))

voting_clf = VotingClassifier(
    estimators=[(name, model) for name, model in best_models.items()],
    voting='soft'
)
voting_clf.fit(X_train, y_train)

y_pred_voting = voting_clf.predict(X_test)
voting_accuracy = accuracy_score(y_test, y_pred_voting)
print(f"Voting Classifier Test Accuracy: {voting_accuracy:.4f}")
print(classification_report(y_test, y_pred_voting))

Logistic Regression: Best parameters: {'classifier__C': 1}
Random Forest: Best parameters: {'classifier__max_depth': None, 'classifier__n_estimators': 50}
Decision Tree: Best parameters: {'classifier__max_depth': None, 'classifier__min_samples_split': 2}


c:\Users\ksshi\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoost: Best parameters: {'classifier__learning_rate': 1, 'classifier__n_estimators': 50}
Logistic Regression Test Accuracy: 0.9038
              precision    recall  f1-score   support

           0       0.83      0.88      0.85        33
           1       0.94      0.92      0.93        71

    accuracy                           0.90       104
   macro avg       0.89      0.90      0.89       104
weighted avg       0.91      0.90      0.90       104

Random Forest Test Accuracy: 0.9904
              precision    recall  f1-score   support

           0       0.97      1.00      0.99        33
           1       1.00      0.99      0.99        71

    accuracy                           0.99       104
   macro avg       0.99      0.99      0.99       104
weighted avg       0.99      0.99      0.99       104

Decision Tree Test Accuracy: 0.9519
              precision    recall  f1-score   support

           0       0.87      1.00      0.93        33
           1       1.00      0.

c:\Users\ksshi\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


In [28]:
os.makedirs('Pickle', exist_ok=True)
pickle_path = os.path.join('Pickle', 'DiabetesClassifier.pkl')
with open(pickle_path, 'wb') as f:
    pickle.dump(voting_clf, f)

print(f"Voting classifier saved as '{pickle_path}'")

Voting classifier saved as 'Pickle\DiabetesClassifier.pkl'
